# T15 — Kinematic feature extraction at subduction zones through time

**Cluster B: Plate kinematics + tectonics.**

*Author-contributed by:* Ehsan Farahbakhsh, adapted from his [`GPlates_Workflows/kinematic_feature_extraction`](https://github.com/e-farahbakhsh/GPlates_Workflows/tree/main/kinematic_feature_extraction) repository, and reimplemented here in `pyGMT` (the original used Cartopy/Matplotlib) to match the suite's house style.

## What this notebook does

Every downstream workflow that scores a subduction zone for something — porphyry-Cu prospectivity (T72, T73, T76, T77), slab-flux budgets (T31), the feature-extractability diagnostics of T19 — starts from the same handful of per-trench-point kinematic quantities: convergence rate, convergence obliquity, trench-orthogonal convergence rate, trench velocity, and flux proxies derived from them. This notebook exposes that shared feature family directly, as a general-purpose diagnostic rather than folded into any one downstream application: at a chosen reconstruction time, it tessellates every subduction zone in the model, computes the full feature set at every trench point, and renders any one feature the user picks as a global paleogeographic map with an interactive time slider.

## Learning objectives

- Use `gplately.PlateReconstruction.tessellate_subduction_zones` to sample per-point kinematics (convergence rate, convergence obliquity, trench-orthogonal rate, trench velocity) along every subduction zone at a reconstruction time.
- Derive simple flux proxies (slab area flux, a lithospheric water-flux proxy) from those kinematic quantities.
- Build an interactive, multi-feature, time-sliding pyGMT map — the same interaction pattern as T07's Panel slider, applied here to a feature-selection dropdown as well as time.
- Recognise this feature family as the shared input layer for the suite's mineral-exploration machine-learning notebooks (T72, T73, T76, T77).

## Prerequisites and runtime

- **Plate model**: Zahirovic 2022 (default; configurable).
- **Time range**: 0–100 Ma in 1 Myr steps (default; configurable).
- **Python**: `gplately`, `pygmt`, `pygplates`, `numpy`, `pandas`, `panel`.
- **Runtime**: ~2–4 minutes for the default 0–100 Ma sweep (subduction-zone tessellation at every step is the slow part).

## Environment + imports

In [ ]:
from pathlib import Path
import os, sys
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import pandas as pd

import pygplates
import gplately
import pygmt
import panel as pn

pn.extension()

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, pd, pygplates, gplately, pygmt, pn):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


## Configuration

In [ ]:
# === USER CONFIGURATION =====================================================
MODEL_NAME          = "Zahirovic2022"
ANCHOR_PLATE_ID     = 0

TIME_MIN_MA         = 0
TIME_MAX_MA         = 100
TIME_STEP_MA        = 5          # coarser default than the upstream 1 Myr — keeps
                                  # the notebook's first run under ~3 minutes;
                                  # tighten to 1 Ma for publication-quality sweeps

DEFAULT_FEATURE     = "convergence_rate (cm/yr)"

# Simple, literature-typical proxies (not a full thermal/petrological model —
# see "Extend this" for how to swap in a more rigorous slab-flux formulation).
# Bulk water content assumed for altered oceanic lithosphere entering the
# trench, after the range summarised in van Keken et al. (2011, JGR) for the
# "subduction factory" water budget.
BULK_WATER_CONTENT_WT_PCT = 2.0
# ============================================================================


## 1. Tessellate subduction zones and compute the kinematic feature set at one time

In [ ]:
from plate_model_manager import PlateModelManager

pmm   = PlateModelManager()
model = pmm.get_model(MODEL_NAME, data_dir="./gplately_data")

recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
    anchor_plate_id=ANCHOR_PLATE_ID,
)

TESS_COLS = ["lon", "lat", "conv_rate", "conv_angle",
             "trench_velocity", "trench_velocity_angle",
             "arc_length", "trench_azimuth_angle",
             "subducting_pid", "trench_pid"]


def compute_kinematic_features(time, recon=recon):
    """Tessellate every subduction zone at `time` and derive the feature set:
    convergence rate/obliquity/orthogonal-rate (direct from GPlately), plus a
    slab area-flux proxy and a lithospheric water-flux proxy (both scaled by
    the trench-orthogonal convergence rate and the local segment length —
    see BULK_WATER_CONTENT_WT_PCT above for the assumption behind the latter).
    Returns a per-point DataFrame, one row per tessellated trench segment."""
    tess = recon.tessellate_subduction_zones(time, ignore_warnings=True)
    df = pd.DataFrame(tess, columns=TESS_COLS)

    conv_rate_orth = df["conv_rate"] * np.abs(np.cos(np.deg2rad(df["conv_angle"])))
    df["convergence_rate (cm/yr)"] = df["conv_rate"]
    df["convergence_obliquity (degrees)"] = df["conv_angle"]
    df["convergence_rate_orthogonal (cm/yr)"] = conv_rate_orth

    # Slab area flux proxy: trench-orthogonal rate x segment arc length.
    df["slab_flux (m^2/yr)"] = (conv_rate_orth * 1e-2) * (df["arc_length"] * 1e3)

    # Lithospheric water-flux proxy: slab area flux x an assumed bulk water
    # content per metre of trench, expressed per unit trench length.
    df["subduction_water_flux_lithosphere (t/m/yr)"] = (
        df["slab_flux (m^2/yr)"] * (BULK_WATER_CONTENT_WT_PCT / 100.0) * 1.0
    )

    df["age (Ma)"] = time
    return df


FEATURES_AVAILABLE = [
    "convergence_rate (cm/yr)",
    "convergence_rate_orthogonal (cm/yr)",
    "convergence_obliquity (degrees)",
    "slab_flux (m^2/yr)",
    "subduction_water_flux_lithosphere (t/m/yr)",
]

_check_df = compute_kinematic_features(0.0)
print(f"{len(_check_df)} trench points tessellated at 0 Ma")
_check_df[FEATURES_AVAILABLE].describe()


### Where these features come from

`convergence_rate`, `convergence_obliquity` (the angle between the convergence vector and the trench-normal), and `trench_velocity` are direct outputs of `tessellate_subduction_zones` — the same routine T19 shows can silently drop points where the kinematics can't be resolved. `slab_flux` and `subduction_water_flux_lithosphere` are simple proxies built on top: an area-flux (trench-orthogonal rate x segment length) and a water-flux estimate scaled by an assumed bulk hydration content for altered oceanic lithosphere (`BULK_WATER_CONTENT_WT_PCT`). These are deliberately simplified for a teaching notebook — see "Extend this" for how to swap in a full thermal/petrological slab-dehydration model.

## 2. Pre-compute the full time series (needed for the interactive slider below)

In [ ]:
TIME_STEPS = list(range(TIME_MIN_MA, TIME_MAX_MA + TIME_STEP_MA, TIME_STEP_MA))

_frames = []
for _t in TIME_STEPS:
    _frames.append(compute_kinematic_features(_t))
subduction_data = pd.concat(_frames, ignore_index=True)

print(f"{len(subduction_data)} total trench-point samples across "
      f"{len(TIME_STEPS)} times ({TIME_MIN_MA}-{TIME_MAX_MA} Ma, "
      f"{TIME_STEP_MA} Myr steps)")


## 3. Interactive multi-feature map (Panel time + feature slider)

In [ ]:
def plot_feature_map(time, feature):
    gplot = gplately.PlotTopologies(
        plate_reconstruction=recon,
        coastlines=model.get_coastlines(),
        continents=model.get_continental_polygons(),
        COBs=model.get_COBs(),
        time=float(time),
        plot_engine=gplately.PygmtPlotEngine(),
    )

    df_t = subduction_data[subduction_data["age (Ma)"] == time]

    if feature == "convergence_obliquity (degrees)":
        vmin, vmax = -90, 90
    else:
        vmin = float(df_t[feature].quantile(0.01))
        vmax = float(df_t[feature].quantile(0.99))
        if feature != "convergence_obliquity (degrees)" and vmin > 0:
            vmin = 0.0

    fig = pygmt.Figure()
    fig.basemap(region="g", projection="W30/16c",
                frame=["af", f'+t{feature} at {time:.0f} Ma'])

    # Continuous-backbone plate-boundary pattern (house style, §3.1).
    gplot.plot_all_topologies(fig, color="gray70", linewidth="0.4p")
    gplot.plot_continents(fig, fill="gray95", pen="0.3p,gray30")
    gplot.plot_coastlines(fig, pen="0.3p,gray30")

    pygmt.makecpt(cmap="batlow", series=[vmin, vmax])
    fig.plot(x=df_t["lon"], y=df_t["lat"], style="c0.1c", fill=feature,
             cmap=True, data=df_t, pen="0.1p,black")
    gplot.plot_trenches(fig, color="black", pen="0.8p")
    gplot.plot_subduction_teeth(fig, color="black", spacing=0.4, size=0.25)

    fig.colorbar(frame=f'af+l"{feature}"', position="JBC+w12c/0.35c+h+o0/1.2c")
    fig.text(text=f"{time:.0f} Ma  ({MODEL_NAME})",
             position="TL", offset="0.25c/-0.25c", justify="TL",
             font="14p,Helvetica-Bold,black", fill="white", pen="0.5p,black")
    return fig


time_slider    = pn.widgets.DiscreteSlider(name="Reconstruction time (Ma)",
                                            options=TIME_STEPS, value=TIME_STEPS[0])
feature_select = pn.widgets.Select(name="Feature", options=FEATURES_AVAILABLE,
                                    value=DEFAULT_FEATURE)

pn.bind(plot_feature_map, time=time_slider, feature=feature_select, watch=False)
pn.Column(pn.Row(time_slider, feature_select),
          pn.bind(lambda t, f: plot_feature_map(t, f).show(), time_slider, feature_select))


### How to read the map

Each dot is one tessellated subduction-zone point at the chosen time, coloured by the selected feature (`batlow`, colour-vision-safe). The reconstructed plate-boundary backbone (light grey), trenches (black, with subduction teeth), and continents give paleogeographic context. Switching the feature dropdown between `convergence_rate` and `convergence_obliquity` at a fixed time is a quick way to see where fast, near-orthogonal subduction (the setting the porphyry-Cu prospectivity notebooks in cluster K single out) is happening at that age.

## Extend this

- **Full slab-flux model.** Swap the area-flux and water-flux proxies here for a proper thermal-plate-age-dependent formulation (e.g. relate `slab_flux` to subducting-plate age via a half-space cooling model, following the general approach in Farahbakhsh's upstream repo) and compare against the global inventory in T31.
- **Feed a prospectivity model.** This notebook's per-point feature table is structurally identical to the input the porphyry-Cu ML notebooks (T72, T73, T76, T77) build — try joining a deposit compilation to the nearest tessellated point at the deposit's mineralisation age (see T72's arc-segment envelope pattern) and training a simple classifier.
- **Different plate model.** Re-run against Cao 2024 for a deep-time (1.8 Ga) view, or Merdith 2021 for the >410 Ma paleomagnetic-frame regime — the feature-extraction machinery is plate-model agnostic.
- **Cross-check against T19.** Run this notebook's `compute_kinematic_features` alongside T19's feature-extractability diagnostic at the same time step to confirm the two agree on where trench points silently drop out.

## References

- Farahbakhsh, E. `GPlates_Workflows` — `kinematic_feature_extraction`. <https://github.com/e-farahbakhsh/GPlates_Workflows/tree/main/kinematic_feature_extraction> (source workflow this notebook is adapted from).
- Mather, B.R., Müller, R.D., Zahirovic, S., Cannon, J., Chin, M., Ilano, L., Wright, N.M., Alfonso, C., Williams, S., Tetley, M., Merdith, A. (2024). Deep time spatio-temporal data analysis using GPlately. *Geoscience Data Journal* 11, 3–10.
- van Keken, P.E., Hacker, B.R., Syracuse, E.M., Abers, G.A. (2011). Subduction factory: 4. Depth-dependent flux of H2O from subducting slabs worldwide. *Journal of Geophysical Research* 116, B01401.
- Tian, D., Uieda, L., Leong, W.J., Fröhlich, Y., Schlitzer, W., Grund, M., Jones, M., Toney, L., Yao, J., Magen, Y., Wessel, P. (2024). PyGMT: A Python interface for the Generic Mapping Tools. *Zenodo*, v0.18.0.